In [40]:
from PyPDF2 import PdfReader 

def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    full_text = ''
    i = 0
    for page in reader.pages:
        i = i + 1
        if i < 10 : 
            text = page.extract_text()
            full_text += text
    return full_text
pdf_text = extract_text_from_pdf("/Users/pushpanjali/citation/Song_Vectorizing_Building_Blueprints_ACCV_2022_paper.pdf")

from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai.embeddings import OpenAIEmbeddings
text_splitter = SemanticChunker(OpenAIEmbeddings(api_key=))
docs = text_splitter.create_documents([pdf_text])

from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(api_key=)
db = FAISS.from_documents(docs, embeddings)


In [41]:
retriever = db.as_retriever()

In [42]:
from langchain.chains import create_history_aware_retriever, create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI, OpenAIEmbeddings


In [43]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.0,
    max_retries=2,
    groq_api_key = "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",
    # other params...
)

In [44]:
from langchain_core.tools import tool
@tool
def document_retriever(query : str) :
        """Function to retrieve documents to generate answer for relevant queries."""
        retriever = db.as_retriever()
        docs = retriever.invoke(query)
        print("=============================") 
        print(docs)
        print("=============================") 

        context_window = f"""
        Document1 : {docs[0].page_content}
        Document2 : {docs[1].page_content}
        Document3 : {docs[2].page_content}
        """
        return context_window


In [45]:
tools = [document_retriever]
tools

[StructuredTool(name='document_retriever', description='Function to retrieve documents to generate answer for relevant queries.', args_schema=<class 'langchain_core.utils.pydantic.document_retriever'>, func=<function document_retriever at 0x14f1c4ea0>)]

In [46]:
llm_with_tools = llm.bind_tools(tools)

In [47]:
contextualize_q_system_prompt = """Given a chat history and the latest user question \
which might reference context in the chat history, formulate a standalone question \
which can be understood without the chat history. Do NOT answer the question, \
just reformulate it if needed and otherwise return it as is."""
contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)
history_aware_retriever = create_history_aware_retriever(
    llm_with_tools, retriever, contextualize_q_prompt
)

In [56]:
history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x14f441970>, search_kwargs={}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChun

In [58]:
qa_system_prompt = """You are an assistant for question-answering tasks.  
You have access to tools that you can use to retrieve relevant context to answer the user query. \
Use the following pieces of retrieved context to answer the question. \ 
If you don't know the answer, just say that you don't know. \
Use three sentences maximum and keep the answer concise.\

{context}"""
# qa_system_prompt = """You are an expert research assistant.Your task is to answer user queries for a document.
#     User queries related to the construction industry, resumes, agreements, contract documents, and related subjects, are to be considered relevant. 
#     Additionally, general queries requesting metadata of a document, such as author name, citation, heading, abstract, or summary, are also considered relevant.  
#     However, queries that are vague, unclear, or pertain to day-to-day communication lines should be considered invalid.
#     If user query is irrelvant output "Please ask document related queries."

#     If user query is relevant, find the quotes from the documents that are most relevant to answering the question, and then print them in numbered order. Quotes should be relatively short.
#     You also have access to tool that you can use to retrieve document chunks to answer the relevant queries whenever required. 
#     If there are no relevant quotes, write “No relevant quotes” instead.

#     Do not generate quotes that are not present in the document write "No relevant quotes" instead

#     Then, answer the question, starting with “Answer:”. Do not include or reference quoted content verbatim in the answer. Don’t say “According to Quote [1]” when answering. Instead make references to quotes relevant to each section of the answer solely by adding their bracketed numbers at the end of relevant sentences.

#     Thus, the format of your overall response should look like what’s shown between the tags. Make sure to follow the formatting and spacing exactly.
#     Quotes:
#     [1] “Company X reported revenue of $12 million in 2021.”
#     [2] “Almost 90% of revenue came from widget sales, with gadget sales making up the remaining 10%.”

#     Answer:
#     Company X earned $12 million. [1] Almost 90% of it was from widget sales. [2]


#     If the question cannot be answered by the documents, say so.
# Use the following pieces of retrieved context to answer the question. 

# {context}"""
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", qa_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)
qa_prompt

<>:1: SyntaxWarning: invalid escape sequence '\ '
<>:1: SyntaxWarning: invalid escape sequence '\ '
/var/folders/s9/gn5z_3cn3sv79r2v3_648np40000gn/T/ipykernel_4948/3377473633.py:1: SyntaxWarning: invalid escape sequence '\ '
  qa_system_prompt = """You are an assistant for question-answering tasks.


ChatPromptTemplate(input_variables=['chat_history', 'context', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.

In [59]:
question_answer_chain = create_stuff_documents_chain(llm_with_tools, qa_prompt)

rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

In [60]:
store = {}


def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

In [61]:
conversational_rag_chain.invoke(
    {"input": "name the authors"},
    config={
        "configurable": {"session_id": "abc123"}
    },  # constructs a key "abc123" in `store`.
)["answer"]



'The authors are Weilian Song, Mahsa Maleki Abyaneh, Mohammad Amin Shabani, and Yasutaka Furukawa.'

In [55]:
conversational_rag_chain.invoke(
    {"input": "what does it tend to miss?"},
    config={
        "configurable": {"session_id": "abc123"}
    },  # constructs a key "abc123" in `store`.
)["answer"]



''